# Muse archive analysis

Explore how your recordings have changed across sessions and over time. The notebook reads the original Muse CSVs and builds session summaries in memory. It runs separately from the synthetic-data and EegML notebooks.

**Run All** to scan the archive and show gamma level, left-versus-right gamma, and the other four EEG bands. The last section shows how the two front channels changed within one recording. The focus section measures the opening rise, sustained gamma and drops for each session.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ARCHIVE = Path.home() / "Documents" / "Yijing" / "Muse"
INSPECT_RECORDING = None  # None selects the most recent usable recording.
SENSOR_PAIR = "Front"  # Use "Rear" for TP9 and TP10 in the other-band plots.

STARTUP_SECONDS = 120
WINDOW_SECONDS = 30
MIN_SECONDS_PER_WINDOW = 24
MIN_WINDOWS_PER_SESSION = 3
MIN_SESSIONS_PER_MONTH = 3

FOCUS_WINDOW_SECONDS = 10
FOCUS_MIN_SECONDS = 8
BASELINE_SECONDS = 30
OPENING_SECONDS = 60
FOCUS_REVIEW_START = "2026-01-01"
FOCUS_GAP_DAYS = 21
WIND_DOWN_SECONDS = 120  # Eyes open and meditation stopped.
DROP_REFERENCE_SECONDS = 60
DROP_DEPTH = 0.25  # Recorded gamma units
DROP_RECOVERY_DEPTH = 0.125

BANDS = ("Delta", "Theta", "Alpha", "Beta", "Gamma")
PAIRS = {"Front": ("AF7", "AF8"), "Rear": ("TP9", "TP10")}
SENSORS = ("AF7", "AF8", "TP9", "TP10")
EEG_COLUMNS = [
    f"{band}_{sensor}" for band in BANDS for sensor in SENSORS
]
GYRO_COLUMNS = ["Gyro_X", "Gyro_Y", "Gyro_Z"]
OPTIONAL_COLUMNS = ["HeadBandOn", *GYRO_COLUMNS]

## How sessions are compared

- Recording dates come from the timestamps inside each file. The first two minutes are omitted because the Bluetooth capture code can use that period as its baseline.
- The analysis takes one median per second, then one median per 30-second window. A window needs at least 24 seconds with readings. Each recording has equal weight in the monthly summaries.
- **Front gamma** is the average of AF7 and AF8 in each paired reading. The **upper gamma** statistic is the 90th percentile of the window medians, so a brief spike carries less weight. **Right minus left** is AF8 minus AF7. Positive values indicate higher readings on the right.
- Plots and early-to-late comparisons separate the earlier Mellow CSV format and the later PPG CSV format. The formats are a proxy for capture settings. The recording code can change the scale or baseline, so compare trends within each format.
- Rows explicitly marked as headset off are omitted. For files with varying gyroscope readings, the summary also reports gamma and right-minus-left after omitting the highest-motion tenth of windows. Constant motion or contact fields provide little information, and low motion alone does not rule out muscle activity.

In [ ]:
def summarize_recording(path):
    """Read one recording and return its session and window summaries."""
    header = pd.read_csv(path, nrows=0).columns
    missing = set(["TimeStamp", *EEG_COLUMNS]) - set(header)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

    if "PPG_IR" in header:
        recording_format = "PPG format"
    elif "Mellow" in header:
        recording_format = "Mellow format"
    else:
        recording_format = "Other format"

    wanted = ["TimeStamp", *EEG_COLUMNS, *OPTIONAL_COLUMNS]
    data = pd.read_csv(path, usecols=[c for c in wanted if c in header], low_memory=False)
    times = pd.to_datetime(data["TimeStamp"], format="mixed", errors="coerce")
    values = (
        data.reindex(columns=[*EEG_COLUMNS, *OPTIONAL_COLUMNS])
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    timed_eeg = times.notna() & values[EEG_COLUMNS].notna().any(axis=1)
    start = times[timed_eeg].min()
    if pd.isna(start):
        raise ValueError("No timestamped EEG readings.")

    elapsed = (times - start).dt.total_seconds()
    headset_off = values["HeadBandOn"].eq(0).fillna(False)
    usable = timed_eeg & ~headset_off
    values = values.loc[usable]
    seconds = elapsed.loc[usable].astype(int)

    signals = {}
    for band in BANDS:
        for pair, (left_sensor, right_sensor) in PAIRS.items():
            left = values[f"{band}_{left_sensor}"]
            right = values[f"{band}_{right_sensor}"]
            signals[f"{band}_{pair}"] = (left + right) / 2
            if band == "Gamma" and pair == "Front":
                paired = left.notna() & right.notna()
                signals["Gamma_AF7"] = left.where(paired)
                signals["Gamma_AF8"] = right.where(paired)
                signals["Gamma_right_minus_left"] = right - left

    signals["Gyro_speed"] = np.sqrt(values[GYRO_COLUMNS].pow(2).sum(axis=1, min_count=3))
    per_second = pd.DataFrame(signals).groupby(seconds).median()
    focus_seconds = per_second[["Gamma_Front", "Gamma_AF7", "Gamma_AF8"]]
    sensor_seconds = values[EEG_COLUMNS].groupby(seconds).median()
    focus_seconds = focus_seconds.join(sensor_seconds.add_prefix("Sensor_"))
    focus_groups = focus_seconds.groupby(focus_seconds.index // FOCUS_WINDOW_SECONDS * FOCUS_WINDOW_SECONDS)
    focus = focus_groups.median().where(focus_groups.count() >= FOCUS_MIN_SECONDS)
    focus.index.name = "Seconds from start"

    per_second = per_second.loc[per_second.index >= STARTUP_SECONDS]
    window_start = STARTUP_SECONDS + (per_second.index - STARTUP_SECONDS) // WINDOW_SECONDS * WINDOW_SECONDS
    grouped = per_second.groupby(window_start)
    windows = grouped.median().where(grouped.count() >= MIN_SECONDS_PER_WINDOW)
    windows.index.name = "Seconds from start"
    windows = windows.dropna(subset=["Gamma_Front"])
    if len(windows) < MIN_WINDOWS_PER_SESSION:
        raise ValueError(f"Only {len(windows)} usable front gamma windows.")

    summary = {
        "File": path.name,
        "Format": recording_format,
        "Started": start,
        "Duration_minutes": elapsed[timed_eeg].max() / 60,
        "Windows": len(windows),
        "Headset_off_rows": int((timed_eeg & headset_off).sum()),
        **windows.drop(columns="Gyro_speed").median().to_dict(),
        "Gamma_upper": windows["Gamma_Front"].quantile(0.9),
    }
    gyro = windows["Gyro_speed"].dropna()
    if len(gyro) >= 10 and gyro.quantile(0.9) > gyro.quantile(0.1):
        quieter = windows.loc[gyro.index[gyro <= gyro.quantile(0.9)]]
        summary["Gamma_lower_motion"] = quieter["Gamma_Front"].median()
        summary["Side_lower_motion"] = quieter["Gamma_right_minus_left"].median()
    return summary, windows, focus

In [ ]:
if not ARCHIVE.is_dir():
    raise FileNotFoundError(f"Set ARCHIVE to your Muse CSV folder. Current path: {ARCHIVE}")

files = sorted(path for path in ARCHIVE.iterdir() if path.suffix.lower() == ".csv" and path.is_file())
summaries, skipped = [], []
focus_by_file = {}
for i, path in enumerate(files, start=1):
    try:
        summary, _, focus = summarize_recording(path)
        focus_by_file[path.name] = focus
        summaries.append(summary)
    except (ValueError, OSError) as error:
        skipped.append({"File": path.name, "Reason": str(error)})
    if i % 50 == 0:
        print(f"Read {i} of {len(files)} recordings.")

if not summaries:
    raise ValueError(f"No usable recordings in {ARCHIVE}. First skipped files: {skipped[:3]}")

sessions = pd.DataFrame(summaries).sort_values("Started", ignore_index=True)
formats = list(sessions["Format"].unique())
print(f"Summarized {len(sessions)} of {len(files)} CSV files.")
display(sessions.groupby("Format", sort=False).agg(
    Sessions=("File", "size"),
    First=("Started", "min"),
    Last=("Started", "max"),
    Median_minutes=("Duration_minutes", "median"),
    Median_windows=("Windows", "median"),
).round({"Median_minutes": 1, "Median_windows": 1}))
if skipped:
    display(pd.DataFrame(skipped))

## Gamma level and left-to-right shift

Each faint dot is a session. A larger dot marks a monthly median where at least three sessions are available. Read the signed AF8 − AF7 panel together with the separate AF7 and AF8 lines to see which side drove the change.

In [ ]:
def plot_trend(ax, group, column, label, color):
    series = group.set_index("Started")[column]
    ax.scatter(series.index, series, color=color, s=10, alpha=0.25)
    monthly = series.resample("MS").agg(["median", "count"])
    enough = monthly["count"] >= MIN_SESSIONS_PER_MONTH
    ax.plot(monthly.index, monthly["median"].where(enough), "o-", color=color,
            markersize=4, label=label)


def format_date_axis(ax):
    locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))


fig, axes = plt.subplots(3, len(formats), figsize=(6 * len(formats), 8),
                         sharex="col", squeeze=False, layout="constrained")
for column, recording_format in enumerate(formats):
    group = sessions.loc[sessions["Format"].eq(recording_format)]
    level, sides, shift = axes[:, column]
    level.set_title(recording_format)
    plot_trend(level, group, "Gamma_Front", "Typical", "tab:blue")
    plot_trend(level, group, "Gamma_upper", "Upper window level", "tab:orange")
    plot_trend(sides, group, "Gamma_AF7", "AF7 · left", "tab:purple")
    plot_trend(sides, group, "Gamma_AF8", "AF8 · right", "tab:green")
    plot_trend(shift, group, "Gamma_right_minus_left", "AF8 − AF7", "tab:brown")
    shift.axhline(0, color="0.4", linewidth=1, linestyle="--")
    format_date_axis(shift)
    for ax, label in zip((level, sides, shift), ("Front gamma", "Gamma by side", "Right minus left")):
        ax.set_ylabel(label)
        ax.grid(axis="y", alpha=0.2)
        ax.legend(loc="best")
plt.show()

## Other EEG bands

Each panel shows session medians for the sensor pair selected in Settings. The bands have their own vertical scales, so compare the direction and size of change within each panel.

In [ ]:
fig, axes = plt.subplots(4, len(formats), figsize=(6 * len(formats), 9),
                         sharex="col", squeeze=False, layout="constrained")
for column, recording_format in enumerate(formats):
    group = sessions.loc[sessions["Format"].eq(recording_format)]
    axes[0, column].set_title(recording_format)
    for ax, band, color in zip(axes[:, column], BANDS[:-1],
                               ("tab:blue", "tab:purple", "tab:green", "tab:orange")):
        plot_trend(ax, group, f"{band}_{SENSOR_PAIR}", f"{SENSOR_PAIR.lower()} pair", color)
        ax.set_ylabel(band)
        ax.grid(axis="y", alpha=0.2)
    format_date_axis(axes[-1, column])
fig.suptitle(f"{SENSOR_PAIR} sensor pair · session medians")
plt.show()

## Early versus late sessions

Compare the median of the first quarter of sessions with the last quarter within each format. These are descriptive changes in recorded units. The date ranges show which periods each comparison covers.

In [ ]:
metrics = {
    "Front gamma": "Gamma_Front",
    "Upper front gamma": "Gamma_upper",
    "Gamma AF7 (left)": "Gamma_AF7",
    "Gamma AF8 (right)": "Gamma_AF8",
    "Gamma right minus left": "Gamma_right_minus_left",
    **{f"{band} ({SENSOR_PAIR.lower()})": f"{band}_{SENSOR_PAIR}" for band in BANDS[:-1]},
}
for recording_format, group in sessions.groupby("Format", sort=False):
    count = len(group) // 4
    if count < 3:
        continue
    early, late = group.head(count), group.tail(count)
    print(f"{recording_format} · {count} sessions per period")
    print(f"Early {early['Started'].min():%Y-%m-%d} to {early['Started'].max():%Y-%m-%d}")
    print(f"Late  {late['Started'].min():%Y-%m-%d} to {late['Started'].max():%Y-%m-%d}")
    comparison = pd.DataFrame({
        "Early": {label: early[column].median() for label, column in metrics.items()},
        "Late": {label: late[column].median() for label, column in metrics.items()},
    })
    comparison["Change"] = comparison["Late"] - comparison["Early"]
    display(comparison.round(3))

if "Gamma_lower_motion" in sessions:
    with_motion = sessions.dropna(subset=["Gamma_lower_motion"]).copy()
    with_motion["Gamma_change_when_quieter"] = with_motion["Gamma_lower_motion"] - with_motion["Gamma_Front"]
    with_motion["Side_change_when_quieter"] = with_motion["Side_lower_motion"] - with_motion["Gamma_right_minus_left"]
    print("Motion check · median change after omitting the highest-motion windows")
    display(with_motion.groupby("Format", sort=False).agg(
        Sessions=("File", "size"),
        Front_gamma=("Gamma_change_when_quieter", "median"),
        Right_minus_left=("Side_change_when_quieter", "median"),
    ).round(3))

## Look inside one recording

Set `INSPECT_RECORDING` in Settings to a filename from the archive. The default uses the most recent usable recording. Each point is one 30-second window.

In [ ]:
selected = INSPECT_RECORDING or sessions.iloc[-1]["File"]
summary, windows, _ = summarize_recording(ARCHIVE / selected)
minutes = (windows.index + WINDOW_SECONDS / 2) / 60
fig, (gamma, side) = plt.subplots(2, 1, sharex=True, figsize=(10, 5), layout="constrained")
fig.suptitle(f"{summary['Started']:%Y-%m-%d %H:%M} · {summary['Format']}")
gamma.plot(minutes, windows["Gamma_AF7"], color="tab:purple", label="AF7 · left")
gamma.plot(minutes, windows["Gamma_AF8"], color="tab:green", label="AF8 · right")
gamma.set_ylabel("Gamma")
gamma.legend()
side.plot(minutes, windows["Gamma_right_minus_left"], color="tab:brown")
side.axhline(0, color="0.4", linewidth=1, linestyle="--")
side.set(xlabel="Minutes from recording start", ylabel="AF8 − AF7")
plt.show()

## Frontal gamma and directed focus

**Working hypothesis.** Directed thought drives my focus, and frontal gamma is its main recorded correlate. I expect gamma to rise as I begin focusing, hold or gradually increase, then drop when I lose focus or briefly doze and rise again as I regain it. The drop markers indicate signal changes. The recordings have no event labels to identify what I was doing at each drop.

Use the **frontal average** (paired AF7 and AF8) as the primary measure. As a sensitivity check, repeat the analysis with the **dominant channel**, defined as whichever side has the higher median after the first two minutes. That side stays fixed for the entire recording, including its baseline. A higher median can reflect a higher recorded level on one side, so compare the patterns across both traces.

- **Meditation end.** The final two minutes are treated as eyes-open wind-down. They stay visible in the full trace as a shaded region. Metrics use only 10-second windows that end by the cutoff.
- **Opening rise.** Subtract the median in the first 30 seconds from the median in minute 2–3. The opening and baseline each require at least two-thirds of their windows. This estimates the net rise relative to an early recording reference. The capture code can subtract a running baseline during the first two minutes, and the actual onset of directed thought is unmarked.
- **Maintenance.** After minute 3, measure the share of valid windows within 0.25 recorded units below the opening level or above it. Also show how the typical post-opening level and the median in the last full minute before the wind-down cutoff differ from the opening level. The final-minute measure requires two-thirds coverage and a final minute entirely after the opening.
- **Drops and recovery.** From minute 3, flag a drop when a 10-second median falls at least 0.25 units below the median of the preceding minute. Keep that reference fixed through the episode and record its maximum depth. Recovery is the first window back within 0.125 units of the reference. A gap or the wind-down cutoff closes an episode with recovery unobserved. The detector requires a full minute of consecutive usable reference windows and never spans missing data. The rate uses only time eligible for detection, including time in an ongoing episode.
- **Across sessions.** Compare dates within each capture format using session dots and monthly medians. Each session has equal weight. Median depth uses sessions with detected drops. Blank values indicate insufficient data or an inapplicable measure.

The thresholds are adjustable in Settings. Ten-second medians emphasize sustained changes and can smooth over shorter dips.

In [ ]:
OPENING_END = STARTUP_SECONDS + OPENING_SECONDS
FOCUS_TRACES = {
    "Frontal average": "tab:blue",
    "Dominant channel": "tab:orange",
}


def interval_level(trace, start, end):
    windows = trace.loc[(trace.index >= start) & (trace.index < end)].dropna()
    if len(windows) * FOCUS_WINDOW_SECONDS >= (end - start) * 2 / 3:
        return windows.median()
    return np.nan


def find_gamma_drops(trace):
    reference_windows = DROP_REFERENCE_SECONDS // FOCUS_WINDOW_SECONDS
    events = []
    eligible_windows = 0
    valid = trace.dropna()
    segments = valid.index.to_series().diff().ne(FOCUS_WINDOW_SECONDS).cumsum()
    for _, segment in valid.groupby(segments):
        event = None
        for offset, (second, level) in enumerate(segment.items()):
            if second < OPENING_END or offset < reference_windows:
                continue
            eligible_windows += 1
            if event is None:
                reference = segment.iloc[offset - reference_windows:offset].median()
                if reference - level >= DROP_DEPTH:
                    event = {
                        "start": second, "reference": reference, "trough": second,
                        "depth": reference - level, "recovered": np.nan,
                    }
                    events.append(event)
            else:
                depth = event["reference"] - level
                if depth <= DROP_RECOVERY_DEPTH:
                    event["recovered"] = second
                    event = None
                elif depth > event["depth"]:
                    event.update(trough=second, depth=depth)
    return events, eligible_windows * FOCUS_WINDOW_SECONDS / 60


def frontal_traces(focus, duration_minutes):
    cutoff = duration_minutes * 60 - WIND_DOWN_SECONDS
    end = int(np.floor(cutoff / FOCUS_WINDOW_SECONDS)) * FOCUS_WINDOW_SECONDS
    meditation = focus.loc[focus.index < end]
    medians = meditation.loc[meditation.index >= STARTUP_SECONDS, ["Gamma_AF7", "Gamma_AF8"]].median().dropna()
    dominant = medians.idxmax() if len(medians) else None
    traces = pd.DataFrame({
        "Frontal average": focus["Gamma_Front"],
        "Dominant channel": focus[dominant] if dominant else np.nan,
    })
    return traces, dominant, end, cutoff


def summarize_focus(focus, duration_minutes):
    traces, dominant, end, cutoff = frontal_traces(focus, duration_minutes)
    rows, events_by_trace = [], {}
    for name, trace in traces.loc[traces.index < end].items():
        baseline = interval_level(trace, 0, BASELINE_SECONDS)
        opening = interval_level(trace, STARTUP_SECONDS, OPENING_END)
        final_start = end - OPENING_SECONDS
        final = interval_level(trace, final_start, end) if final_start >= OPENING_END else np.nan
        sustained = trace.loc[trace.index >= OPENING_END].dropna()
        hold = np.nan
        if sustained.size and pd.notna(opening):
            hold = 100 * sustained.ge(opening - DROP_DEPTH).mean()
        events, eligible_minutes = find_gamma_drops(trace)
        depths = pd.Series([event["depth"] for event in events], dtype=float)
        recovery_seconds = pd.Series(
            [event["recovered"] - event["start"] for event in events if pd.notna(event["recovered"])],
            dtype=float,
        )
        events_by_trace[name] = events
        rows.append({
            "Trace": name,
            "Channels": "AF7 + AF8" if name == "Frontal average" else (dominant or "").removeprefix("Gamma_"),
            "Baseline": baseline,
            "Opening": opening,
            "Opening rise": opening - baseline,
            "Typical minus opening": sustained.median() - opening,
            "End of meditation minus opening": final - opening,
            "Time holding (%)": hold,
            "Eligible minutes": eligible_minutes,
            "Drops": len(events) if eligible_minutes else np.nan,
            "Drops per 10 min": 10 * len(events) / eligible_minutes if eligible_minutes else np.nan,
            "Median drop depth": depths.median() if events else np.nan,
            "Deepest drop": depths.max() if events else np.nan,
            "Recoveries observed": len(recovery_seconds) if eligible_minutes else np.nan,
            "Observed recovery (%)": 100 * len(recovery_seconds) / len(events) if events else np.nan,
            "Median observed recovery (s)": recovery_seconds.median(),
        })
    return traces, rows, events_by_trace, cutoff


focus_rows = []
for session in sessions[["File", "Started", "Format", "Duration_minutes"]].to_dict("records"):
    _, rows, _, _ = summarize_focus(focus_by_file[session["File"]], session.pop("Duration_minutes"))
    focus_rows.extend({**session, **row} for row in rows)
focus_sessions = pd.DataFrame(focus_rows)

In [ ]:
traces, _, events_by_trace, cutoff = summarize_focus(focus_by_file[selected], summary["Duration_minutes"])
selected_focus = focus_sessions.loc[focus_sessions["File"].eq(selected)].set_index("Trace")
display(selected_focus.drop(columns=["File", "Started", "Format"]).round(3))

# Reindex so missing windows appear as breaks in the plotted lines.
traces = traces.reindex(range(0, traces.index.max() + FOCUS_WINDOW_SECONDS, FOCUS_WINDOW_SECONDS))
midpoint = FOCUS_WINDOW_SECONDS / 2
minutes = (traces.index + midpoint) / 60
fig, axes = plt.subplots(2, 2, figsize=(13, 6), sharey="row", layout="constrained",
                         gridspec_kw={"width_ratios": [1, 2]})
fig.suptitle(f"{summary['Started']:%Y-%m-%d %H:%M} · {summary['Format']}")
axes[0, 0].set_title("Opening detail")
axes[0, 1].set_title("Full recording")
for (name, color), (detail, full) in zip(FOCUS_TRACES.items(), axes):
    metrics = selected_focus.loc[name]
    for ax in (detail, full):
        ax.plot(minutes, traces[name], color=color, linewidth=1.2)
        ax.axvspan(0, BASELINE_SECONDS / 60, color="0.5", alpha=0.14, label="Baseline")
        ax.axvspan(STARTUP_SECONDS / 60, OPENING_END / 60, color=color, alpha=0.13, label="Opening")
        if pd.notna(metrics["Opening"]):
            ax.axhline(metrics["Opening"], color=color, linestyle="--", alpha=0.6)
        ax.grid(axis="y", alpha=0.2)
    detail.set(xlim=(0, OPENING_END / 60), ylabel=f"{name}\nGamma")
    detail.legend(loc="best")
    if WIND_DOWN_SECONDS > 0:
        full.axvspan(max(0, cutoff / 60), summary["Duration_minutes"], color="0.5", alpha=0.23,
                     label="Eyes open / wind-down")
    events = events_by_trace[name]
    for i, event in enumerate(events):
        trough = (event["trough"] + midpoint) / 60
        level = traces.loc[event["trough"], name]
        full.vlines(trough, level, event["reference"], color="tab:red", alpha=0.65)
        full.scatter(trough, level, color="tab:red", s=20, zorder=3,
                     label="Drop depth" if i == 0 else "_nolegend_")
    recovered = [event["recovered"] for event in events if pd.notna(event["recovered"])]
    if recovered:
        full.scatter((np.asarray(recovered) + midpoint) / 60, traces.loc[recovered, name],
                     marker="^", color="tab:green", s=22, zorder=3, label="Recovery")
    if events or WIND_DOWN_SECONDS > 0:
        handles, labels = full.get_legend_handles_labels()
        full.legend(handles[2:], labels[2:], loc="best")
for ax in axes[-1]:
    ax.set_xlabel("Minutes from recording start")
plt.show()

## Peak and average frontal gamma over time

The combined trace averages paired AF7 and AF8 readings. The dominant trace uses whichever frontal sensor has the higher median during the session and keeps that sensor fixed throughout. Both traces use 10-second medians and exclude the two-minute startup and eyes-open wind-down.

- **Peak** is the highest valid 10-second median. **Session average** is the arithmetic mean of all valid 10-second medians in the session.
- **Best 2, 5 and 10 minutes** are the highest rolling averages over those durations. The window advances in 10-second steps and requires a complete run of valid readings. Short sessions and gaps can leave a value blank.
- Faint dots are individual sessions. Larger markers are monthly medians, shown where at least three sessions in a recording format have the metric. Colour identifies recording format. The two traces share the same vertical scale within each row.

In [ ]:
best_window_minutes = (2, 5, 10)
gamma_high_metrics = ["Peak (10 s)", "Session average"] + [
    f"Best {minutes} min average" for minutes in best_window_minutes
]


def gamma_level_metrics(trace):
    regular = trace.reindex(range(STARTUP_SECONDS, trace.index.max() + FOCUS_WINDOW_SECONDS,
                                  FOCUS_WINDOW_SECONDS)) if len(trace) else trace
    metrics = {"Peak (10 s)": regular.max(), "Session average": regular.mean()}
    for minutes in best_window_minutes:
        windows = minutes * 60 // FOCUS_WINDOW_SECONDS
        metrics[f"Best {minutes} min average"] = regular.rolling(windows).mean().max()
    return metrics


gamma_high_rows = []
for session in sessions[["File", "Format", "Started", "Duration_minutes"]].to_dict("records"):
    traces, _, end, _ = frontal_traces(focus_by_file[session["File"]], session.pop("Duration_minutes"))
    for name, trace in traces.loc[(traces.index >= STARTUP_SECONDS) & (traces.index < end)].items():
        gamma_high_rows.append({**session, "Trace": name, **gamma_level_metrics(trace)})
gamma_session_highs = pd.DataFrame(gamma_high_rows)

fig, axes = plt.subplots(len(gamma_high_metrics), len(FOCUS_TRACES), figsize=(13, 11),
                         sharex=True, sharey="row", layout="constrained")
fig.suptitle("Peak and average frontal gamma over time")
for column, name in enumerate(FOCUS_TRACES):
    trace_sessions = gamma_session_highs.loc[gamma_session_highs["Trace"].eq(name)]
    axes[0, column].set_title("Combined AF7 and AF8" if name == "Frontal average" else name)
    for row, metric in enumerate(gamma_high_metrics):
        ax = axes[row, column]
        for recording_format, group in trace_sessions.groupby("Format", sort=False):
            color = {"Mellow format": "tab:blue", "PPG format": "tab:orange"}.get(
                recording_format, "tab:green"
            )
            plot_trend(ax, group, metric, recording_format, color)
        if column == 0:
            ax.set_ylabel(metric)
        ax.grid(axis="y", alpha=0.2)
axes[0, 0].legend(loc="best")
format_date_axis(axes[-1, 0])
plt.show()

## Focus metrics across recordings

Compare the typical level and the end-of-meditation level against the opening level to see whether the ending reflects the rest of the session. Read those changes alongside time holding. Drop count shows how many episodes occurred, and the rate accounts for differences in usable recording time. Median depth shows the typical episode, and deepest drop shows the most pronounced one. Depth and level changes use recorded gamma units.

In [ ]:
for title, metrics in (
    ("Opening rise and maintenance", ("Opening rise", "Typical minus opening", "End of meditation minus opening", "Time holding (%)")),
    ("Drops and recovery", ("Drops", "Drops per 10 min", "Median drop depth", "Deepest drop",
                            "Recoveries observed")),
):
    fig, axes = plt.subplots(len(metrics), len(formats), figsize=(5 * len(formats), 2.4 * len(metrics)),
                             sharex="col", sharey="row", squeeze=False, layout="constrained")
    fig.suptitle(title)
    for column, recording_format in enumerate(formats):
        group = focus_sessions.loc[focus_sessions["Format"].eq(recording_format)]
        axes[0, column].set_title(recording_format)
        for ax, metric in zip(axes[:, column], metrics):
            for name, color in FOCUS_TRACES.items():
                plot_trend(ax, group.loc[group["Trace"].eq(name)], metric, name, color)
            ax.set_ylabel(metric.replace("End of meditation minus opening", "Meditation end\nminus opening"))
            ax.grid(axis="y", alpha=0.2)
            if metric in ("Opening rise", "Typical minus opening", "End of meditation minus opening"):
                ax.axhline(0, color="0.4", linewidth=1, linestyle="--")
            if metric == "Time holding (%)":
                ax.set_ylim(0, 103)
        axes[0, column].legend(loc="best")
        format_date_axis(axes[-1, column])
    plt.show()

archive_table = focus_sessions.drop(columns=["Baseline", "Opening"])
precision = dict.fromkeys(archive_table.select_dtypes("number").columns, 3)
display(archive_table.round(precision))

## The 2026 shift in the retained archive

Headset failures account for the major recording gaps. Meditation generally continued during those gaps. Many earlier short or weak recordings were deleted, so the retained early sessions are a selected sample. The archive cannot quantify how the deleted sessions would change the comparison.

The timeline shows frontal averages from 2026 onward on common axes. Colour identifies recording format, the dashed line marks the first PPG recording, and shading spans gaps of at least 21 days between retained recordings. Each monthly marker is a median and its vertical bar spans the middle half of sessions in that month and format. Months need at least three valid sessions for a summary.

The distribution plots compare pre-2026 Mellow, 2026 Mellow and 2026 PPG recordings. A curve gives the fraction of sessions at or below a given value. Look at the low end of time holding and the high end of drop rate to see whether weaker sessions became less common. The January boundary is a comparison date. The distributions use retained sessions and weight them equally.

Raw recovery counts depend on how many drops there were. The observed recovery percentage uses detected drops as its denominator and omits sessions with zero drops. Gaps and the wind-down cutoff can leave recovery unobserved. Recovery time runs from detection to the first window meeting the recovery threshold and uses observed recoveries only. Eligible minutes show how much of a session contributed to the drop rate.

In [ ]:
primary = focus_sessions.loc[focus_sessions["Trace"].eq("Frontal average")].copy()
review_start = pd.Timestamp(FOCUS_REVIEW_START)
recent = primary.loc[primary["Started"].ge(review_start)]
format_colors = {"Mellow format": "tab:blue", "PPG format": "tab:orange", "Other format": "tab:green"}
review_metrics = {
    "Time holding (%)": "Time holding (%)",
    "Drops per 10 min": "Drops per 10 min",
    "Observed recovery (%)": "Drops with observed recovery (%)",
    "Median observed recovery (s)": "Median recovery time (s)",
    "Eligible minutes": "Eligible minutes",
}
dates = primary["Started"].drop_duplicates().sort_values()
gaps = [(start, end) for start, end in zip(dates.iloc[:-1], dates.iloc[1:])
        if end >= review_start and end - start >= pd.Timedelta(days=FOCUS_GAP_DAYS)]
first_ppg = primary.loc[primary["Format"].eq("PPG format"), "Started"].min()

if not recent.empty:
    fig, axes = plt.subplots(len(review_metrics), 1, sharex=True, figsize=(12, 11), layout="constrained")
    fig.suptitle(f"Frontal gamma since {review_start:%b %Y}")
    for ax, (metric, label) in zip(axes, review_metrics.items()):
        for i, (start, end) in enumerate(gaps):
            ax.axvspan(max(start, review_start), end, color="0.9",
                       label="Recording gap" if i == 0 else "_nolegend_")
        if pd.notna(first_ppg) and first_ppg >= review_start:
            ax.axvline(first_ppg, color="0.35", linestyle="--", label="First PPG recording")
        for recording_format, group in recent.groupby("Format", sort=False):
            values = group.set_index("Started")[metric].dropna()
            color = format_colors[recording_format]
            ax.scatter(values.index, values, s=15, alpha=0.35, color=color, label=recording_format)
            monthly = values.resample("MS").agg(
                count="size", median="median", lower=lambda s: s.quantile(.25), upper=lambda s: s.quantile(.75),
            )
            monthly = monthly.loc[monthly["count"].ge(MIN_SESSIONS_PER_MONTH)]
            center = monthly.index + pd.Timedelta(days=14)
            ax.errorbar(center, monthly["median"],
                        yerr=[monthly["median"] - monthly["lower"], monthly["upper"] - monthly["median"]],
                        fmt="o", color=color, capsize=3, markersize=4)
        ax.set_ylabel(label.replace("with observed ", "with observed\n"))
        ax.grid(axis="y", alpha=0.2)
    axes[0].legend(ncols=2, loc="best")
    axes[-1].set_xlim(left=review_start)
    format_date_axis(axes[-1])
    plt.show()

comparison = primary.loc[primary["Format"].isin(["Mellow format", "PPG format"])].copy()
comparison["Period"] = np.where(
    comparison["Started"].lt(review_start), f"Before {review_start:%b %Y}", f"From {review_start:%b %Y}"
) + " · " + comparison["Format"].str.removesuffix(" format")
groups = list(comparison.groupby("Period", sort=False))
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True, layout="constrained")
fig.suptitle("Retained session distributions · frontal average")
for ax, (metric, label) in zip(axes.flat, list(review_metrics.items())[:4]):
    for (period, group), color in zip(groups, ("tab:purple", "tab:blue", "tab:orange", "tab:green")):
        values = group[metric].dropna().sort_values().to_numpy()
        if values.size:
            fraction = np.arange(1, values.size + 1) / values.size
            ax.step(np.r_[values[0], values], np.r_[0, fraction], where="post",
                    color=color, label=f"{period} (n={values.size})")
    ax.set_xlabel(label)
    ax.grid(alpha=0.2)
for ax in axes[:, 0]:
    ax.set_ylabel("Fraction of sessions at or below")
axes[0, 0].legend(loc="best")
plt.show()

display(comparison.groupby("Period", sort=False).agg(
    Sessions=("File", "size"),
    Holding_lower_quartile=("Time holding (%)", lambda s: s.quantile(.25)),
    Holding_median=("Time holding (%)", "median"),
    Drop_rate_median=("Drops per 10 min", "median"),
    Drop_rate_upper_quartile=("Drops per 10 min", lambda s: s.quantile(.75)),
    Recovery_percent_median=("Observed recovery (%)", "median"),
    Recovery_seconds_median=("Median observed recovery (s)", "median"),
    Eligible_minutes_median=("Eligible minutes", "median"),
).round(2))

## How session duration changed

Duration is measured from the end of the two-minute startup to the start of the eyes-open wind-down. It is elapsed time, including any missing EEG windows within those bounds. A recording needs a positive interval to contribute.

Each weekly or monthly point is the arithmetic mean duration per retained session that started in that period. Weeks begin on Monday. Small grey dots show individual sessions, and the lower panels show how many contributed to each mean. Periods with no recordings are left blank. The shading marks gaps between retained recordings, and the dashed line marks the first PPG recording. Periods spanning the format change can include both formats.

The Yijing Divination target and trigger system largely determines when a session ends and is sensitive to focus and drops. Duration is therefore a measure of how long these sessions ran under that system. Read it alongside time holding and drop rate. Each scatter point is one session, and colour shows the recording format and period. Drop rate uses eligible EEG time as its denominator, which can be shorter than the elapsed duration on the vertical axis. The retained archive omits deleted earlier sessions and periods when the headset was unavailable.

In [ ]:
duration_column = "Session minutes"
duration_sessions = sessions[["File", "Started", "Format", "Duration_minutes"]].copy()
duration_sessions[duration_column] = duration_sessions["Duration_minutes"] - (
    STARTUP_SECONDS + WIND_DOWN_SECONDS
) / 60
duration_sessions = duration_sessions.loc[duration_sessions[duration_column].gt(0)]
if duration_sessions.empty:
    print("No recordings have time between startup and wind-down.")
else:
    duration_dates = sessions["Started"].sort_values()
    duration_gaps = [(start, end) for start, end in zip(duration_dates.iloc[:-1], duration_dates.iloc[1:])
                     if end - start >= pd.Timedelta(days=FOCUS_GAP_DAYS)]
    fig, axes = plt.subplots(2, 2, sharex="col", sharey="row", figsize=(14, 6),
                             layout="constrained", height_ratios=[3, 1])
    fig.suptitle("Session duration across the retained archive")
    duration_periods = {}
    for (label, frequency), (mean_ax, count_ax) in zip(
        (("Weekly", "W-SUN"), ("Monthly", "M")), axes.T
    ):
        periods = duration_sessions["Started"].dt.to_period(frequency)
        grouped = duration_sessions.groupby(periods)[duration_column].agg(Mean="mean", Sessions="size")
        period_index = pd.period_range(periods.min(), periods.max(), freq=frequency)
        grouped = grouped.reindex(period_index)
        duration_periods[label] = grouped
        starts = period_index.start_time
        widths = (period_index.end_time + pd.Timedelta(1, unit="ns") - starts).days
        centers = starts + pd.to_timedelta(widths / 2, unit="D")
        for i, (start, end) in enumerate(duration_gaps):
            for ax in (mean_ax, count_ax):
                ax.axvspan(start, end, color="0.93", zorder=0,
                           label="Recording gap" if i == 0 else "_nolegend_")
        if pd.notna(first_ppg):
            for ax in (mean_ax, count_ax):
                ax.axvline(first_ppg, color="0.35", linestyle="--", label="First PPG")
        mean_ax.scatter(duration_sessions["Started"], duration_sessions[duration_column],
                        s=9, color="0.45", alpha=0.35, label="Session")
        mean_ax.plot(centers, grouped["Mean"], "o-", color="tab:blue", markersize=3, label="Mean")
        count_ax.bar(centers, grouped["Sessions"], width=widths * 0.85, color="tab:blue", alpha=0.7)
        mean_ax.set_title(label)
        for ax in (mean_ax, count_ax):
            ax.grid(axis="y", alpha=0.2)
        format_date_axis(count_ax)
    axes[0, 0].set_ylabel("Minutes per session")
    axes[1, 0].set_ylabel("Sessions")
    axes[0, 0].legend(loc="best", ncols=2)
    plt.show()

    duration_focus = duration_sessions.merge(
        primary[["File", "Time holding (%)", "Drops per 10 min"]], on="File", validate="one_to_one"
    )
    duration_focus["Period"] = np.where(
        duration_focus["Started"].lt(review_start), f"Before {review_start:%b %Y}", f"From {review_start:%b %Y}"
    ) + " · " + duration_focus["Format"].str.removesuffix(" format")
    fig, axes = plt.subplots(1, 2, sharey=True, figsize=(12, 4.5), layout="constrained")
    fig.suptitle("Session duration and frontal gamma")
    for ax, metric in zip(axes, ("Time holding (%)", "Drops per 10 min")):
        for period, group in duration_focus.groupby("Period", sort=False):
            ax.scatter(group[metric], group[duration_column], s=18, alpha=0.55, label=period)
        ax.set_xlabel(metric)
        ax.grid(alpha=0.2)
    axes[0].set_ylabel("Minutes per session")
    axes[1].legend(loc="best")
    plt.show()

    display(duration_focus.groupby("Period", sort=False)[duration_column].agg(
        Sessions="size", Mean="mean", Median="median",
    ).round(1))

## Band and sensor comparisons

These plots use the gamma band values as recorded in the CSV, with the same 10-second windows as the focus analysis. The separate RAW columns are waveform samples. Gamma recomputed from those samples would be a separate analysis.

- **Levels and changes.** Each heatmap compares gamma at one sensor with bands at all four sensors. Compare the same-sensor beta and gamma cells with beta at the other sensors. The within-session Spearman correlations use at least 30 paired windows. Adjacent changes use consecutive windows only. The plot reports the median of session correlations, separately by recording format, and requires at least five eligible sessions per cell. The boxed cells compare channels at the same sensor.
- **Holding.** Compare each band and sensor between post-opening windows that meet the frontal gamma holding threshold and those below it. The difference is expressed in units of that channel’s within-session interquartile range (IQR). Each state needs at least three readings, and a cell needs at least five sessions. Positive values mean the signal is higher during holding. Sessions without enough below-hold windows are omitted from this comparison.
- **Drop and recovery.** Align each band and sensor to frontal-average drop detection and observed recovery. Each event uses the preceding 60 to 30 seconds as its reference and requires a continuous 90-second span around the marker. Changes are in units of the channel’s session IQR. Take the median across events within each session, then the median across sessions. Rear channels use dashed lines. Legends give the number of sessions contributing a complete curve.

Compare levels with adjacent changes to see whether co-movement is visible on a 10-second scale as well as over the course of a session. Same-sensor co-movement can be consistent with band bleed. These recordings alone cannot distinguish it from shared signal, artifact or capture processing. The events are defined by frontal gamma, so the frontal gamma curves show the selection built into the analysis. All comparisons exclude startup and wind-down, and holding and event curves use the post-opening period.

In [ ]:
sensor_columns = [f"Sensor_{band}_{sensor}" for band in BANDS for sensor in SENSORS]
gamma_columns = [f"Sensor_Gamma_{sensor}" for sensor in SENSORS]
session_correlations, hold_contrasts, event_curves = [], [], []
event_offsets = np.arange(-60, 40, FOCUS_WINDOW_SECONDS)

for file, recording_format, duration in sessions[["File", "Format", "Duration_minutes"]].itertuples(index=False, name=None):
    focus = focus_by_file[file]
    cutoff = duration * 60 - WIND_DOWN_SECONDS
    meditation = focus.loc[(focus.index >= STARTUP_SECONDS) &
                           (focus.index + FOCUS_WINDOW_SECONDS <= cutoff)]
    levels = meditation[sensor_columns]
    changes = levels.diff().where(levels.index.to_series().diff().eq(FOCUS_WINDOW_SECONDS), axis=0)
    for kind, values in (("Levels", levels), ("Adjacent changes", changes)):
        correlation = values.corr(method="spearman", min_periods=30).loc[gamma_columns]
        session_correlations.append(correlation.assign(Format=recording_format, Measure=kind))

    post = meditation.loc[meditation.index >= OPENING_END]
    bands = post[sensor_columns]
    scale = bands.quantile(.75) - bands.quantile(.25)
    scale = scale.where(scale.gt(0))
    opening = interval_level(meditation["Gamma_Front"], STARTUP_SECONDS, OPENING_END)
    if pd.notna(opening):
        gamma = post["Gamma_Front"]
        holding = gamma.ge(opening - DROP_DEPTH)
        below = gamma.lt(opening - DROP_DEPTH)
        high, low = bands.loc[holding], bands.loc[below]
        enough = high.count().ge(3) & low.count().ge(3)
        contrast = ((high.median() - low.median()) / scale).where(enough)
        hold_contrasts.append(pd.DataFrame([contrast], columns=sensor_columns).assign(Format=recording_format))

    events, _ = find_gamma_drops(meditation["Gamma_Front"])
    for label, key in (("Drop detection", "start"), ("Observed recovery", "recovered")):
        curves = []
        for event in events:
            if pd.isna(event[key]):
                continue
            aligned = bands.reindex(event[key] + event_offsets)
            valid_gamma = post["Gamma_Front"].reindex(aligned.index).notna().all()
            if not valid_gamma:
                continue
            baseline = aligned.loc[event_offsets <= -30].median()
            complete = aligned.notna().all()
            curve = ((aligned - baseline) / scale).where(complete, axis=1)
            curve.index = event_offsets
            curves.append(curve)
        if curves:
            median_curve = pd.concat(curves).groupby(level=0).median()
            event_curves.append(median_curve.assign(Format=recording_format, Event=label))

band_correlations = pd.concat(session_correlations)
band_holding = pd.concat(hold_contrasts, ignore_index=True) if hold_contrasts else pd.DataFrame()
band_events = pd.concat(event_curves) if event_curves else pd.DataFrame()

In [ ]:
from matplotlib.patches import Rectangle

for recording_format in formats:
    fig, axes = plt.subplots(3, 1, figsize=(13, 6.5), sharex=True, layout="constrained",
                             height_ratios=[4, 4, 1])
    fig.suptitle(f"Band and sensor comparisons · {recording_format}")
    for ax, measure in zip(axes[:2], ("Levels", "Adjacent changes")):
        group = band_correlations.loc[band_correlations["Format"].eq(recording_format) &
                                      band_correlations["Measure"].eq(measure), sensor_columns]
        by_gamma = group.groupby(level=0)
        matrix = by_gamma.median().where(by_gamma.count().ge(5)).reindex(gamma_columns)
        heat = ax.imshow(matrix, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
        ax.set_yticks(range(len(SENSORS)), [f"Gamma {sensor}" for sensor in SENSORS])
        ax.set_title(measure, loc="left")
        for row, sensor in enumerate(SENSORS):
            for column in range(len(sensor_columns)):
                value = matrix.iloc[row, column]
                if pd.notna(value):
                    ax.text(column, row, f"{value:.2f}", ha="center", va="center",
                            fontsize=8, color="white" if abs(value) > .63 else "black")
                if SENSORS[column % len(SENSORS)] == sensor:
                    ax.add_patch(Rectangle((column - .5, row - .5), 1, 1, fill=False, linewidth=.8))
    fig.colorbar(heat, ax=list(axes[:2]), label="Median Spearman correlation", shrink=.85)
    hold_ax = axes[-1]
    contrast = pd.Series(np.nan, index=sensor_columns)
    if not band_holding.empty:
        group = band_holding.loc[band_holding["Format"].eq(recording_format), sensor_columns]
        contrast = group.median().where(group.count().ge(5))
    limit = max(1, contrast.abs().max()) if contrast.notna().any() else 1
    hold_heat = hold_ax.imshow([contrast.to_numpy()], vmin=-limit, vmax=limit, cmap="RdBu_r", aspect="auto")
    hold_ax.set_yticks([0], ["Holding"])
    hold_ax.set_xticks(range(len(sensor_columns)),
                       [f"{band}\n{sensor}" for band in BANDS for sensor in SENSORS], fontsize=8)
    for column, value in enumerate(contrast):
        if pd.notna(value):
            hold_ax.text(column, 0, f"{value:.2f}", ha="center", va="center",
                         fontsize=8, color="white" if abs(value) > .63 * limit else "black")
    fig.colorbar(hold_heat, ax=hold_ax, label="IQR units")
    plt.show()

if not band_events.empty:
    for recording_format in formats:
        group = band_events.loc[band_events["Format"].eq(recording_format)]
        if group.empty:
            continue
        fig, axes = plt.subplots(len(BANDS), 2, sharex=True, sharey="row", figsize=(12, 11),
                                 layout="constrained")
        fig.suptitle(f"Signals around frontal gamma events · {recording_format}")
        for column, label in enumerate(("Drop detection", "Observed recovery")):
            event_group = group.loc[group["Event"].eq(label), sensor_columns].groupby(level=0)
            counts = event_group.count().reindex(event_offsets).fillna(0).min()
            medians = event_group.median()
            axes[0, column].set_title(label)
            for ax, band in zip(axes[:, column], BANDS):
                for sensor, color in zip(SENSORS, ("tab:blue", "tab:cyan", "tab:orange", "tab:red")):
                    name = f"Sensor_{band}_{sensor}"
                    if counts[name] >= 5:
                        ax.plot(medians.index, medians[name], color=color,
                                linestyle="-" if sensor in PAIRS["Front"] else "--",
                                label=f"{sensor} (n={int(counts[name])})")
                ax.axvline(0, color=".45", linewidth=1)
                ax.axhline(0, color=".75", linewidth=1)
                if ax.get_legend_handles_labels()[0]:
                    ax.legend(fontsize=8, ncols=2)
                if column == 0:
                    ax.set_ylabel(f"{band}\nIQR units")
            axes[-1, column].set_xlabel("Seconds relative to marker")
        plt.show()